# Detection Rate Calculator

In [1]:
# --- Installation & Setup (Colab only) ---

import sys

# Check if the env run under Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Clone our GitHub repository into the Colab environment
    !git clone https://github.com/weizmannk/cbc-population-distributions.git
    %cd cbc-population-distributions

    # Install uv (fast dependency manager) and sync dependencies from uv.lock
    !pip install -q uv
    !uv sync

    # (Optional) Add the repo to PYTHONPATH if it's not an installable package

    import glob
    import os

    pyver = f"{sys.version_info.major}.{sys.version_info.minor}"
    venv_site = glob.glob(f".venv/lib/python{pyver}/site-packages")[0]
    sys.path.insert(0, os.path.abspath(venv_site))

    print("Environment ready. You can now run the rest of the notebook.")

## Imports and Setup

In [2]:
import logging

import numpy as np
from astropy import units as u
from astropy.table import Table
from IPython.display import Markdown, display

# Import Poisson utility functions
from poisson_rate_utils import format_with_errorbars, poisson_lognormal_rate_quantiles
from scipy import stats

print("Imports successful")

Imports successful


In [3]:
logging.basicConfig(
    level=logging.INFO,
    format="[%(levelname)s] %(message)s",
    stream=sys.stdout,
    force=True,
)

---

## 1. Function Definitions

In [4]:
def build_rates_table(fiducial_rates, mass_fractions):
    """
    Build a table with fiducial rates adjusted by mass fractions.

    Parameters
    ----------
    fiducial_rates : dict
        Dictionary with population names as keys and rate dicts as values
    mass_fractions : dict
        Dictionary with population names as keys and fractions as values

    Returns
    -------
    rates_table : astropy.table.Table
        Table with adjusted rates
    """
    rows = []
    for pop in ["BNS", "NSBH", "BBH"]:
        row = {"population": pop}
        row.update(fiducial_rates[pop])
        rows.append(row)

    rates_table = Table(rows)

    # Add mass fractions
    rates_table["mass_fraction"] = [
        mass_fractions["BNS"],
        mass_fractions["NSBH"],
        mass_fractions["BBH"],
    ]

    # Adjust rates by mass fractions
    for key in ["lower", "mid", "upper"]:
        rates_table[key] *= rates_table["mass_fraction"]

    # Calculate log-normal parameters (mu and sigma)
    # Assuming log-normal distribution where lower=5% and upper=95%
    rates_table["fiducial_log_rate"] = np.log(rates_table["mid"])

    # For log-normal: 90% interval
    standard_90pct_interval = np.diff(stats.norm.interval(0.9))[0]

    rates_table["fiducial_log_rate_err"] = (
        np.log(rates_table["upper"]) - np.log(rates_table["lower"])
    ) / standard_90pct_interval

    return rates_table


def calculate_detection_rates(
    run_name, run_config, rates_table, prob_quantiles=[0.05, 0.5, 0.95]
):
    """
    Calculate detection rates for a specific observing run.

    Parameters
    ----------
    run_name : str
        Name of the observing run
    run_config : dict
        Dictionary with 'duration', 'sim_rate', and 'detections'
    rates_table : astropy.table.Table
        Table with fiducial rates and parameters
    prob_quantiles : list
        Probability quantiles [lower, median, upper]

    Returns
    -------
    results : dict
        Dictionary with 'low', 'mid', 'high' estimates for each population
    """
    # Convert simulation rate to Gpc^-3 yr^-1
    sim_rate_gpc = (run_config["sim_rate"] / (u.Mpc**3 * u.yr)).to(u.Gpc**-3 * u.yr**-1)

    results = {"low": {}, "mid": {}, "high": {}}

    for pop in ["BNS", "NSBH", "BBH"]:
        # Get rate info for this population
        rates_row = rates_table[rates_table["population"] == pop]

        # Effective rate for this population
        rate = sim_rate_gpc * rates_row["mass_fraction"][0]

        # Calculate mu for this run
        detection_count = run_config["detections"][pop]
        mu = (
            rates_row["fiducial_log_rate"][0]
            + np.log(run_config["duration"])
            + np.log(detection_count / rate.value)
        )
        sigma = rates_row["fiducial_log_rate_err"][0]

        # Calculate quantiles using Poisson-lognormal distribution
        lo, mid, hi = poisson_lognormal_rate_quantiles(
            np.asarray(prob_quantiles), mu, sigma
        )

        # Round
        lo = int(np.floor(lo))
        mid = int(np.round(mid))
        hi = int(np.ceil(hi))

        # Format with error bars
        mid_str, lo_str, hi_str = format_with_errorbars(mid, lo, hi)

        results["low"][pop] = lo_str
        results["mid"][pop] = mid_str
        results["high"][pop] = hi_str

    return results


def format_results_table(all_results, observing_runs):
    """
    Format all results into a display table.

    Parameters
    ----------
    all_results : dict
        Dictionary of results for each run
    observing_runs : dict
        Dictionary of observing run configurations

    Returns
    -------
    results_table : astropy.table.Table
        Formatted table for display
    """
    rows = []
    for run_name, run_stats in all_results.items():
        for pop in ["BNS", "NSBH", "BBH"]:
            rows.append(
                {
                    "run": run_name,
                    "population": pop,
                    "detections": observing_runs[run_name]["detections"][pop],
                    "rate_low": f"-{run_stats['low'][pop]}",
                    "rate_mid": run_stats["mid"][pop],
                    "rate_high": f"+{run_stats['high'][pop]}",
                }
            )

    return Table(
        rows=rows,
        names=("run", "population", "detections", "rate_low", "rate_mid", "rate_high"),
    )


def display_results(results_table, observing_runs):
    """
    Display results in a formatted way.

    Parameters
    ----------
    results_table : astropy.table.Table
        Table with all results
    observing_runs : dict
        Dictionary of observing run configurations
    """
    print("\n" + "=" * 80)
    print("DETECTION RATE RESULTS")
    print("=" * 80)

    for run_name in sorted(set(results_table["run"])):
        run_config = observing_runs[run_name]
        run_data = results_table[results_table["run"] == run_name]

        print(f"\n{'─' * 80}")
        print(f" Run: {run_name}")
        print(f" Duration: {run_config['duration']} years")
        print(f" Sim Rate: {run_config['sim_rate']:.2e} Mpc⁻³ yr⁻¹")
        print(f"{'─' * 80}")
        print(f"{'Population':<12} {'Detected':<12} {'Annual Rate (90% CI)':<30}")
        print(f"{'─' * 80}")

        for row in run_data:
            rate_str = f"{row['rate_mid']} ({row['rate_low']}, {row['rate_high']})"
            print(f"{row['population']:<12} {row['detections']:<12} {rate_str:<30}")

        print(f"{'─' * 80}")

    print("\n" + "=" * 80)
    print("Calculation complete!")
    print("=" * 80 + "\n")


print("Functions defined successfully")

Functions defined successfully


---

## 2. Configuration

### 2.1 Mass Classification Parameters

In [5]:
# Mass threshold for neutron star classification (Solar masses)
ns_max_mass = 3.0

display(Markdown(rf"Mass threshold: {ns_max_mass} $M_\odot$"))
display(Markdown(f"- **BNS**: both masses < {ns_max_mass} $M_\\odot$"))
display(
    Markdown(
        f"- **NSBH**: one mass $\geq$ {ns_max_mass} $M_\\odot$, one < {ns_max_mass} $M_\\odot$"
    )
)
display(Markdown(f"- **BBH**: both masses $\geq$ {ns_max_mass} $M_\\odot$"))

Mass threshold: 3.0 $M_\odot$

- **BNS**: both masses < 3.0 $M_\odot$

- **NSBH**: one mass $\geq$ 3.0 $M_\odot$, one < 3.0 $M_\odot$

- **BBH**: both masses $\geq$ 3.0 $M_\odot$

### 2.2 Fiducial rate parameters (90% credible intervals in Gpc^-3 yr^-1)

 Table 2. Merger rates from GWTC-4 catalog in units Gpc−3, https://arxiv.org/pdf/2508.18083

In [6]:
# Fiducial rates: 90% credible intervals (Gpc^-3 yr^-1)
fiducial_rates = {
    "BNS": {"lower": 50.0, "mid": 130.0, "upper": 290.0},
    "NSBH": {"lower": 50.0, "mid": 130.0, "upper": 290.0},
    "BBH": {"lower": 50.0, "mid": 130.0, "upper": 290.0},
}

display(Markdown("Fiducial merger rates $(Gpc^{-3} yr^{-1}$)"))
for pop, rates in fiducial_rates.items():
    logging.info(
        f"  {pop:4s}: {rates['mid']:.0f} ({rates['lower']:.0f} - {rates['upper']:.0f})"
    )

Fiducial merger rates $(Gpc^{-3} yr^{-1}$)

[INFO]   BNS : 130 (50 - 290)
[INFO]   NSBH: 130 (50 - 290)
[INFO]   BBH : 130 (50 - 290)


### 2.3 Mass Fractions (from CBC population model)

In [7]:
# Mass fractions from CBC PB2P model
# Based on 1 million total CBCs
mass_fractions = {"BNS": 738291 / 1e6, "NSBH": 162187 / 1e6, "BBH": 99522 / 1e6}

total_fraction = sum(mass_fractions.values())
logging.info("Mass fractions from CBC population:")
for pop, frac in mass_fractions.items():
    logging.info(f"  {pop:4s}: {frac:.4f} ({frac * 100:.2f}%)")
logging.info(f" Total: {total_fraction:.4f}")

if abs(total_fraction - 1.0) > 0.01:
    logging.info("\n Warning: Mass fractions don't sum to 1.0")

[INFO] Mass fractions from CBC population:
[INFO]   BNS : 0.7383 (73.83%)
[INFO]   NSBH: 0.1622 (16.22%)
[INFO]   BBH : 0.0995 (9.95%)
[INFO]  Total: 1.0000


### 2.4 Observing Runs Configuration

In observing scenarios data there is a file sqlite  file "sqlite3 events.sqlite" where we can read the 

the simulation Merger rate, for that use this command line in terminal or import sqlite with python 

 Use this command to retrieve comments:

 1- $ sqlite3 events.sqlite

 2- $ select comment from process;


In [8]:
observing_runs = {
    "O4a_GWTC-4": {
        "duration": 0.75,  #  Run duration in years,
        "sim_rate": 7.2624879649601604e-06,  # Mpc^-3 yr^-1
        "detections": {"BNS": 234, "NSBH": 282, "BBH": 3304},
    }
}

# Display configured runs
print(f" Configured {len(observing_runs)} observing run(s):\n")
for run_name, config in observing_runs.items():
    total_det = sum(config["detections"].values())
    print(f"  {run_name}:")
    print(f"    Duration: {config['duration']} years")
    print(f"    Sim rate: {config['sim_rate']:.2e} Mpc⁻³ yr⁻¹")
    print(f"    Total detections: {total_det}")
    print(
        f"      BNS={config['detections']['BNS']}, "
        f"NSBH={config['detections']['NSBH']}, "
        f"BBH={config['detections']['BBH']}"
    )

 Configured 1 observing run(s):

  O4a_GWTC-4:
    Duration: 0.75 years
    Sim rate: 7.26e-06 Mpc⁻³ yr⁻¹
    Total detections: 3820
      BNS=234, NSBH=282, BBH=3304


---

## 3. Run the dtection rate

3.1 Build Rates Table

In [9]:
# Build the rates table
print("Building rates table...")
rates_table = build_rates_table(fiducial_rates, mass_fractions)

print("\nAdjusted Rates Table:")
print("=" * 80)
print(
    rates_table[
        [
            "population",
            "mid",
            "mass_fraction",
            "fiducial_log_rate",
            "fiducial_log_rate_err",
        ]
    ]
)
print("=" * 80)

Building rates table...

Adjusted Rates Table:
population   mid    mass_fraction fiducial_log_rate  fiducial_log_rate_err
---------- -------- ------------- ------------------ ---------------------
       BNS 95.97783      0.738291  4.564117227297666    0.5343508652530804
      NSBH 21.08431      0.162187 3.0485291619772634    0.5343508652530805
       BBH 12.93786      0.099522  2.560157896725406    0.5343508652530804


### 3.2 Calculate detection rates for all Runs

In [10]:
# Calculate detection rates for all configured runs
print("\nCalculating detection rates...\n")

all_results = {}
for run_name, run_config in observing_runs.items():
    print(f"  Processing {run_name}...", end=" ")

    all_results[run_name] = calculate_detection_rates(run_name, run_config, rates_table)

    print("")

print("\n All calculations complete!")


Calculating detection rates...

  Processing O4a_GWTC-4... 

 All calculations complete!


---

## 4. Results

### 4.1 Create Results Table

In [11]:
# Format results into a table
results_table = format_results_table(all_results, observing_runs)

print("\nResults Table:")
print("=" * 80)
print(results_table)
print("=" * 80)


Results Table:
   run     population detections rate_low rate_mid rate_high
---------- ---------- ---------- -------- -------- ---------
O4a_GWTC-4        BNS        234       -2        2        +6
O4a_GWTC-4       NSBH        282       -3        3        +7
O4a_GWTC-4        BBH       3304      -28       44       +64


### 4.2 Display Formatted Results

In [12]:
# Display detailed results
display_results(results_table, observing_runs)


DETECTION RATE RESULTS

────────────────────────────────────────────────────────────────────────────────
 Run: O4a_GWTC-4
 Duration: 0.75 years
 Sim Rate: 7.26e-06 Mpc⁻³ yr⁻¹
────────────────────────────────────────────────────────────────────────────────
Population   Detected     Annual Rate (90% CI)          
────────────────────────────────────────────────────────────────────────────────
BNS          234          2 (-2, +6)                    
NSBH         282          3 (-3, +7)                    
BBH          3304         44 (-28, +64)                 
────────────────────────────────────────────────────────────────────────────────

Calculation complete!



### 5.3 Export Results (Optional)

In [13]:
# # Export to CSV
# output_file = "detection_rates_results.csv"
# results_table.write(output_file, format="csv", overwrite=True)
# print(f"\n Results exported to: {output_file}")

# For Multible distribustion and different Fiducial

In [14]:
# The run duration in years,
duration = 0.75

# Define multiple fiducial rate scenarios
fiducial_rate_scenarios = {
    "GWTC-3": {
        "BNS": {"lower": 100.0, "mid": 240.0, "upper": 510.0},
        "NSBH": {"lower": 100.0, "mid": 240.0, "upper": 510.0},
        "BBH": {"lower": 100.0, "mid": 240.0, "upper": 510.0},
    },
    "GWTC-4": {
        "BNS": {"lower": 50.0, "mid": 130.0, "upper": 290.0},
        "NSBH": {"lower": 50.0, "mid": 130.0, "upper": 290.0},
        "BBH": {"lower": 50.0, "mid": 130.0, "upper": 290.0},
    },
}

# Define mass fraction scenarios
mass_fraction_scenarios = {
    "GWTC-3": {"BNS": 892762 / 1e6, "NSBH": 35962 / 1e6, "BBH": 71276 / 1e6},
    "GWTC-4": {"BNS": 738291 / 1e6, "NSBH": 162187 / 1e6, "BBH": 99522 / 1e6},
}


observing_runs = {
    "O4a_GWTC-3": {
        "duration": duration,  # The run duration in years,
        "sim_rate": 8.209138761890435e-06,
        "detections": {"BNS": 405, "NSBH": 90, "BBH": 4222},
        "scenario": "GWTC-3",
    },
    "O4a_GWTC-4": {
        "duration": duration,  # The run duration in years,
        "sim_rate": 7.2624879649601604e-06,
        "detections": {"BNS": 234, "NSBH": 282, "BBH": 3304},
        "scenario": "GWTC-4",
    },
    "O4a_GWTC-3_snr-10_mesurePSD": {
        "duration": duration,  # The run duration in years,
        "sim_rate": 9.761353524686725e-06,
        "detections": {"BNS": 239, "NSBH": 42, "BBH": 2459},
        "scenario": "GWTC-3",
    },
}

In [15]:
all_results = {}
for run_name, run_config in observing_runs.items():
    print(f"  Processing {run_name}...", end=" ")

    # Get the appropriate scenario
    scenario = run_config["scenario"]

    # Build rates table for this scenario
    rates_table = build_rates_table(
        fiducial_rate_scenarios[scenario], mass_fraction_scenarios[scenario]
    )

    # Calculate detection rates
    all_results[run_name] = calculate_detection_rates(run_name, run_config, rates_table)

  Processing O4a_GWTC-3...   Processing O4a_GWTC-4...   Processing O4a_GWTC-3_snr-10_mesurePSD... 

In [16]:
# Format ALL results into a single table
results_table = format_results_table(all_results, observing_runs)

print("\nResults Table:")
print("=" * 80)
print(results_table)
print("=" * 80)


Results Table:
            run             population detections rate_low rate_mid rate_high
--------------------------- ---------- ---------- -------- -------- ---------
                 O4a_GWTC-3        BNS        405       -6        8       +14
                 O4a_GWTC-3       NSBH         90       -1        1        +4
                 O4a_GWTC-3        BBH       4222      -54       92      +119
                 O4a_GWTC-4        BNS        234       -2        2        +6
                 O4a_GWTC-4       NSBH        282       -3        3        +7
                 O4a_GWTC-4        BBH       3304      -28       44       +64
O4a_GWTC-3_snr-10_mesurePSD        BNS        239       -4        4        +7
O4a_GWTC-3_snr-10_mesurePSD       NSBH         42       -0        0        +0
O4a_GWTC-3_snr-10_mesurePSD        BBH       2459      -28       45       +59


In [17]:
# Display detailed results for all runs
display_results(results_table, observing_runs)


DETECTION RATE RESULTS

────────────────────────────────────────────────────────────────────────────────
 Run: O4a_GWTC-3
 Duration: 0.75 years
 Sim Rate: 8.21e-06 Mpc⁻³ yr⁻¹
────────────────────────────────────────────────────────────────────────────────
Population   Detected     Annual Rate (90% CI)          
────────────────────────────────────────────────────────────────────────────────
BNS          405          8 (-6, +14)                   
NSBH         90           1 (-1, +4)                    
BBH          4222         92 (-54, +119)                
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
 Run: O4a_GWTC-3_snr-10_mesurePSD
 Duration: 0.75 years
 Sim Rate: 9.76e-06 Mpc⁻³ yr⁻¹
────────────────────────────────────────────────────────────────────────────────
Population   Detected     Annual Rate (90% CI)          
──────────────────────────────────────────────

In [18]:
output_file = "detection_rates_results.csv"
results_table.write(output_file, format="csv", overwrite=True)
print(f"\n Results exported to: {output_file}")


 Results exported to: detection_rates_results.csv


In [19]:
# import matplotlib.pyplot as plt
# from matplotlib.colors import LinearSegmentedColormap

# # High-quality settings
# plt.rcParams["font.family"] = "sans-serif"
# plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica"]
# plt.rcParams["font.size"] = 11
# plt.rcParams["axes.linewidth"] = 1.5

# # Data
# data = {
#     "GWTC-3": {
#         "BNS": {"mid": 8, "low": 6, "high": 14},
#         "NSBH": {"mid": 1, "low": 1, "high": 4},
#         "BBH": {"mid": 92, "low": 54, "high": 119},
#     },
#     "GWTC-4": {
#         "BNS": {"mid": 2, "low": 2, "high": 6},
#         "NSBH": {"mid": 3, "low": 3, "high": 7},
#         "BBH": {"mid": 44, "low": 28, "high": 64},
#     },
#     "Observed": {"BNS": 0, "NSBH": 1, "BBH": 84},
# }

# populations = ["BNS", "NSBH", "BBH"]
# colors = {"BNS": "#e74c3c", "NSBH": "#3498db", "BBH": "#2ecc71"}
# markers = {"GWTC-3": "o", "GWTC-4": "s"}


# def create_concordance_heatmap():
#     fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

#     # Calculate agreement metrics
#     models = ["GWTC-3", "GWTC-4"]

#     # Matrix: rows = populations, cols = models
#     # Value = 1 - |Obs - Pred| / max(Obs, Pred)
#     agreement_matrix = np.zeros((len(populations), len(models)))
#     ratio_matrix = np.zeros((len(populations), len(models)))

#     for i, pop in enumerate(populations):
#         obs = data["Observed"][pop]
#         for j, model in enumerate(models):
#             pred = data[model][pop]["mid"]

#             # Agreement score (0 to 1, 1 = perfect)
#             if obs == 0 and pred == 0:
#                 agreement = 1.0
#             elif obs == 0 or pred == 0:
#                 agreement = 0.0
#             else:
#                 agreement = 1 - abs(obs - pred) / max(obs, pred)

#             # Ratio
#             ratio = obs / pred if pred > 0 else 0

#             agreement_matrix[i, j] = agreement
#             ratio_matrix[i, j] = ratio

#     # Plot 1: Agreement heatmap
#     im1 = ax1.imshow(
#         agreement_matrix,
#         cmap="RdYlGn",
#         aspect="auto",
#         vmin=0,
#         vmax=1,
#         interpolation="nearest",
#     )

#     # Add text annotations
#     for i in range(len(populations)):
#         for j in range(len(models)):
#             text = ax1.text(
#                 j,
#                 i,
#                 f"{agreement_matrix[i, j]:.2f}",
#                 ha="center",
#                 va="center",
#                 color="black",
#                 fontsize=14,
#                 fontweight="bold",
#             )

#     ax1.set_xticks(range(len(models)))
#     ax1.set_xticklabels(models, fontsize=12, fontweight="bold")
#     ax1.set_yticks(range(len(populations)))
#     ax1.set_yticklabels(populations, fontsize=12, fontweight="bold")
#     ax1.set_title(
#         "Agreement Score\n(1 = Perfect)", fontweight="bold", fontsize=13, pad=10
#     )

#     cbar1 = plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
#     cbar1.set_label("Agreement", fontweight="bold", fontsize=11)

#     # Plot 2: Ratio heatmap
#     # Custom colormap: red (<1), white (1), blue (>1)
#     colors_custom = ["#d62728", "#ff9999", "#ffffff", "#9999ff", "#1f77b4"]
#     n_bins = 100
#     cmap_ratio = LinearSegmentedColormap.from_list(
#         "custom_ratio", colors_custom, N=n_bins
#     )

#     im2 = ax2.imshow(
#         ratio_matrix,
#         cmap=cmap_ratio,
#         aspect="auto",
#         vmin=0,
#         vmax=2.5,
#         interpolation="nearest",
#     )

#     # Add text annotations
#     for i in range(len(populations)):
#         for j in range(len(models)):
#             ratio_val = ratio_matrix[i, j]
#             color = "white" if 0.3 < ratio_val < 1.7 else "black"
#             text = ax2.text(
#                 j,
#                 i,
#                 f"{ratio_val:.2f}",
#                 ha="center",
#                 va="center",
#                 color=color,
#                 fontsize=14,
#                 fontweight="bold",
#             )

#     ax2.set_xticks(range(len(models)))
#     ax2.set_xticklabels(models, fontsize=12, fontweight="bold")
#     ax2.set_yticks(range(len(populations)))
#     ax2.set_yticklabels(populations, fontsize=12, fontweight="bold")
#     ax2.set_title(
#         "Obs./Pred. Ratio\n(1 = Perfect)", fontweight="bold", fontsize=13, pad=10
#     )

#     cbar2 = plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
#     cbar2.set_label("Ratio", fontweight="bold", fontsize=11)

#     plt.suptitle("Model Concordance Analysis", fontweight="bold", fontsize=16, y=0.98)
#     plt.tight_layout(rect=[0, 0, 1, 0.96])

#     return fig


# if __name__ == "__main__":
#     # Figure 3: Heatmap
#     fig3 = create_concordance_heatmap()
#     plt.show()

In [20]:
# ============================================================================
# Load and process data
# ============================================================================
Farah = Table.read("injections-HL_mesured-PSD_SNR_1O.dat", format="ascii.fast_tab")
ns_max_mass = 3.0

# ============================================================================
# Print population statistics
# ============================================================================
print(
    f"Number of BNS: {len(Farah[(Farah['mass1'] < ns_max_mass)])}, "
    f"Number of NSBH: {len(Farah[(Farah['mass1'] >= ns_max_mass) & (Farah['mass2'] < ns_max_mass)])}, "
    f"Number of BBH: {len(Farah[(Farah['mass2'] >= ns_max_mass)])}"
)

Number of BNS: 239, Number of NSBH: 42, Number of BBH: 2459
